In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries loaded successfully!")

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "Dataviz_proj_all_datasets.xlsx").exists():
    ROOT = ROOT.parent

PROJECT_ROOT = ROOT
WORKBOOK = PROJECT_ROOT / "Dataviz_proj_all_datasets.xlsx"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

sys.path.insert(0, str(SCRIPTS_DIR))

from build_dashboard_data import read_workbook_tables, prepare_tables

raw_tables, source_label = read_workbook_tables(WORKBOOK)
tables = prepare_tables(raw_tables)

sellers = tables["sellers"]
order_items = tables["items"]
orders = tables["orders"]
reviews = tables["reviews"]

print("Project root:", PROJECT_ROOT)
print("Loaded source:", source_label)
print("All four seller analysis datasets loaded successfully!")


In [ ]:
print("Sellers:", sellers.shape)
print("Order Items:", order_items.shape)
print("Orders:", orders.shape)
print("Reviews:", reviews.shape)

Which sellers contribute the most marketplace value in terms of revenue and unique orders?

In [ ]:
print("========== ORDER ITEMS ==========")

print("Columns:")
print(order_items.columns.tolist())

print("\nFirst 5 rows:")
display(order_items.head())

print("\nMissing values:")
display(order_items.isna().sum())

In [ ]:
print("Total order-item rows:", len(order_items))
print("Unique sellers:", order_items['seller_id'].nunique())
print("Unique orders:", order_items['order_id'].nunique())

print("\nOrders with multiple item rows:",
      (order_items['order_id'].value_counts() > 1).sum())

print("\nSellers with multiple orders:",
      (order_items['seller_id'].value_counts() > 1).sum())

In [ ]:
seller_business = (
    order_items
    .groupby('seller_id')
    .agg(
        revenue=('price', 'sum'),
        order_count=('order_id', 'nunique'),
        item_count=('order_item_id', 'count'),
        freight_value=('freight_value', 'sum')
    )
    .reset_index()
)

print("Seller-level table created.")
print("Number of sellers:", len(seller_business))

display(seller_business.head())

In [ ]:
print("========== SELLER BUSINESS SUMMARY ==========")

display(
    seller_business[
        ['revenue', 'order_count', 'item_count', 'freight_value']
    ].describe()
)

**Seller contribution is highly uneven. The median seller generated 6 unique orders and R$821.48 in product revenue, while the highest seller generated 1,854 orders and R$229,472.63 in revenue. This suggests that seller performance should be evaluated using both contribution and customer-experience metrics, rather than treating all sellers equally.**

In [ ]:
print("========== TOP 10 SELLERS BY REVENUE ==========")

display(
    seller_business
    .sort_values('revenue', ascending=False)
    .head(10)
)

Seller  contribution finding: The highest-revenue sellers contribute substantially different numbers of orders. For example, the second-highest revenue seller generated approximately R$222.8K from 358 orders, while another seller generated approximately R$200.5K from 1,806 orders. This indicates that seller contribution should be evaluated using both revenue and order volume before identifying strategic or operational priorities.

In [ ]:
top_revenue = (
    seller_business
    .sort_values('revenue', ascending=False)
    .head(10)
)

print("========== TOP 10 SELLERS BY REVENUE ==========")

display(
    top_revenue[
        ['seller_id', 'revenue', 'order_count',
         'item_count', 'freight_value']
    ]
)

In [ ]:
top_orders = (
    seller_business
    .sort_values('order_count', ascending=False)
    .head(10)
)

print("========== TOP 10 SELLERS BY UNIQUE ORDERS ==========")

display(
    top_orders[
        ['seller_id', 'revenue', 'order_count',
         'item_count', 'freight_value']
    ]
)

Seller contribution varies substantially by both order volume and revenue. High-volume sellers do not necessarily generate the highest revenue. Therefore, seller performance should be evaluated using multiple contribution metrics rather than order count or revenue alone.

In [ ]:
seller_business_geo = seller_business.merge(
    sellers[
        ['seller_id',
         'seller_city',
         'seller_state',
         'seller_zip_code_prefix']
    ],
    on='seller_id',
    how='left'
)

print("Seller business + geography table:")
print(seller_business_geo.shape)

display(seller_business_geo.head())

In [ ]:
print("Missing seller states:",
      seller_business_geo['seller_state'].isna().sum())

print("\nNumber of seller states:",
      seller_business_geo['seller_state'].nunique())

print("\nSeller distribution by state:")
display(
    seller_business_geo['seller_state']
    .value_counts()
    .head(15)
)

Seller geography is highly concentrated. São Paulo accounts for approximately 60% of sellers (1,849 of 3,095), while several states have very small seller populations. Therefore, geographic delivery comparisons should consider seller/order volume before drawing conclusions.

**Which sellers have delivery problems?**

In [ ]:
date_columns = [
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors='coerce'
    )

print("Date columns converted successfully.")

In [ ]:
orders['delivery_delay_days'] = (
    orders['order_delivered_customer_date']
    - orders['order_estimated_delivery_date']
).dt.total_seconds() / (24 * 60 * 60)

orders['delivery_status'] = np.where(
    orders['order_delivered_customer_date'].isna(),
    'Not Delivered',
    np.where(
        orders['delivery_delay_days'] <= 0,
        'Early / On-time',
        'Late'
    )
)

display(
    orders[
        ['order_id',
         'order_delivered_customer_date',
         'order_estimated_delivery_date',
         'delivery_delay_days',
         'delivery_status']
    ].head(10)
)

In [ ]:
seller_orders = (
    order_items[['order_id', 'seller_id']]
    .drop_duplicates()
)

print("Seller-order combinations:", len(seller_orders))
print("Unique sellers:", seller_orders['seller_id'].nunique())
print("Unique orders:", seller_orders['order_id'].nunique())

display(seller_orders.head())

In [ ]:
seller_order_delivery = seller_orders.merge(
    orders[
        [
            'order_id',
            'order_delivered_customer_date',
            'order_estimated_delivery_date',
            'delivery_delay_days',
            'delivery_status'
        ]
    ],
    on='order_id',
    how='left'
)

print("Seller-order delivery rows:", len(seller_order_delivery))

print("\nMissing delivery status:",
      seller_order_delivery['delivery_status'].isna().sum())

display(seller_order_delivery.head())

In [ ]:
seller_delivery = (
    seller_order_delivery
    .groupby('seller_id')
    .agg(
        total_seller_orders=('order_id', 'nunique'),
        delivered_orders=(
            'order_delivered_customer_date',
            lambda x: x.notna().sum()
        ),
        late_orders=(
            'delivery_status',
            lambda x: (x == 'Late').sum()
        ),
        not_delivered_orders=(
            'delivery_status',
            lambda x: (x == 'Not Delivered').sum()
        ),
        avg_delivery_delay_days=(
            'delivery_delay_days',
            'mean'
        )
    )
    .reset_index()
)

# Late rate among delivered orders
seller_delivery['late_rate'] = np.where(
    seller_delivery['delivered_orders'] > 0,
    seller_delivery['late_orders']
    / seller_delivery['delivered_orders'] * 100,
    np.nan
)

# Not-delivered rate among all seller orders
seller_delivery['not_delivered_rate'] = (
    seller_delivery['not_delivered_orders']
    / seller_delivery['total_seller_orders'] * 100
)

print("Seller delivery table created:")
print(seller_delivery.shape)

display(seller_delivery.head())

In [ ]:
print("========== SELLER ORDER VOLUME ==========")

display(
    seller_delivery['total_seller_orders'].describe()
)

In [ ]:
print("========== DELIVERED ORDER VOLUME ==========")

display(
    seller_delivery['delivered_orders'].describe()
)

In [ ]:
thresholds = [5, 10, 20, 50, 100]

for threshold in thresholds:
    count = (
        seller_delivery['delivered_orders'] >= threshold
    ).sum()

    print(
        f"At least {threshold} delivered orders: "
        f"{count} sellers"
    )

Minimum 20 delivered orders for seller performance comparisons.

In [ ]:
MIN_DELIVERED_ORDERS = 20

seller_delivery_filtered = seller_delivery[
    seller_delivery['delivered_orders'] >= MIN_DELIVERED_ORDERS
].copy()

print("Minimum delivered orders:", MIN_DELIVERED_ORDERS)
print("Sellers included:", len(seller_delivery_filtered))
print("Sellers excluded:", len(seller_delivery) - len(seller_delivery_filtered))

display(
    seller_delivery_filtered[
        ['seller_id',
         'total_seller_orders',
         'delivered_orders',
         'late_orders',
         'late_rate',
         'not_delivered_rate']
    ].head()
)

In [ ]:
print("========== LATE RATE DISTRIBUTION ==========")

display(
    seller_delivery_filtered['late_rate'].describe()
)

In [ ]:
seller_reviews = (
    seller_order_delivery[
        ['seller_id', 'order_id']
    ]
    .merge(
        reviews[
            ['order_id', 'review_score']
        ],
        on='order_id',
        how='left'
    )
)

print("Seller-review rows:", len(seller_reviews))
print("Orders with review:", seller_reviews['review_score'].notna().sum())
print("Orders without review:", seller_reviews['review_score'].isna().sum())

display(seller_reviews.head())

In [ ]:
order_review_clean = (
    reviews
    .groupby('order_id', as_index=False)
    .agg(
        review_score=('review_score', 'mean')
    )
)

print("Original review rows:", len(reviews))
print("Unique reviewed orders:", order_review_clean['order_id'].nunique())

display(order_review_clean.head())

In [ ]:
print("Review score range:",
      order_review_clean['review_score'].min(),
      "to",
      order_review_clean['review_score'].max())

print("\nReview score distribution:")
display(
    order_review_clean['review_score']
    .value_counts()
    .sort_index()
)

In [ ]:
seller_satisfaction = (
    seller_order_delivery[
        ['seller_id', 'order_id']
    ]
    .drop_duplicates()
    .merge(
        order_review_clean,
        on='order_id',
        how='left'
    )
    .groupby('seller_id')
    .agg(
        reviewed_orders=('review_score', lambda x: x.notna().sum()),
        avg_review_score=('review_score', 'mean'),
        low_review_count=('review_score', lambda x: (x <= 2).sum())
    )
    .reset_index()
)

seller_satisfaction['low_review_rate'] = np.where(
    seller_satisfaction['reviewed_orders'] > 0,
    seller_satisfaction['low_review_count']
    / seller_satisfaction['reviewed_orders'] * 100,
    np.nan
)

print("Seller satisfaction table:", seller_satisfaction.shape)

display(seller_satisfaction.head())

In [ ]:
print("========== SELLER SATISFACTION DISTRIBUTION ==========")

display(
    seller_satisfaction[
        [
            'reviewed_orders',
            'avg_review_score',
            'low_review_rate'
        ]
    ].describe()
)

In [ ]:
seller_master = (
    seller_business_geo
    .merge(
        seller_delivery,
        on='seller_id',
        how='left'
    )
    .merge(
        seller_satisfaction,
        on='seller_id',
        how='left'
    )
)

print("========== SELLER MASTER TABLE ==========")
print("Rows:", len(seller_master))
print("Columns:", len(seller_master.columns))

display(seller_master.head())

In [ ]:
print("========== SELLER REVENUE DISTRIBUTION ==========")

display(
    seller_master['revenue'].describe()
)

In [ ]:
HIGH_REVENUE_THRESHOLD = seller_master['revenue'].quantile(0.75)

high_revenue_count = (
    seller_master['revenue'] > HIGH_REVENUE_THRESHOLD
).sum()

print("High-revenue threshold:", HIGH_REVENUE_THRESHOLD)
print("Sellers above threshold:", high_revenue_count)

In [ ]:
HIGH_LATE_RATE = seller_delivery_filtered['late_rate'].quantile(0.75)

high_revenue_and_late = seller_master[
    (seller_master['revenue'] > HIGH_REVENUE_THRESHOLD) &
    (seller_master['delivered_orders'] >= MIN_DELIVERED_ORDERS) &
    (seller_master['late_rate'] > HIGH_LATE_RATE)
]

print("High late-rate threshold:", HIGH_LATE_RATE)
print("High-revenue + high-late sellers:", len(high_revenue_and_late))

In [ ]:
print("========== HIGH-VALUE + HIGH-LATE SELLERS ==========")

display(
    high_revenue_and_late[
        [
            'seller_id',
            'revenue',
            'delivered_orders',
            'late_orders',
            'late_rate',
            'reviewed_orders',
            'avg_review_score',
            'low_review_rate',
            'seller_state'
        ]
    ]
    .sort_values(
        ['late_rate', 'revenue'],
        ascending=[False, False]
    )
    .head(20)
)

In [ ]:
eligible_sellers = seller_master[
    (seller_master['delivered_orders'] >= MIN_DELIVERED_ORDERS) &
    seller_master['low_review_rate'].notna()
].copy()

MIN_REVIEWED_ORDERS = 20
eligible_sellers = eligible_sellers[
    eligible_sellers['reviewed_orders'] >= MIN_REVIEWED_ORDERS
].copy()

HIGH_LOW_REVIEW_RATE = (
    eligible_sellers['low_review_rate']
    .quantile(0.75)
)

print("Minimum reviewed orders:", MIN_REVIEWED_ORDERS)
print("Sellers included in review threshold:", len(eligible_sellers))
print("High low-review-rate threshold:", HIGH_LOW_REVIEW_RATE)


In [ ]:
intervention_watchlist = seller_master[
    (seller_master['revenue'] > HIGH_REVENUE_THRESHOLD) &
    (seller_master['delivered_orders'] >= MIN_DELIVERED_ORDERS) &
    (seller_master['late_rate'] > HIGH_LATE_RATE) &
    (seller_master['low_review_rate'] > HIGH_LOW_REVIEW_RATE)
].copy()

print("========== INTERVENTION WATCHLIST ==========")
print("Number of sellers:", len(intervention_watchlist))

display(
    intervention_watchlist[
        [
            'seller_id',
            'revenue',
            'delivered_orders',
            'late_rate',
            'reviewed_orders',
            'avg_review_score',
            'low_review_rate',
            'seller_state'
        ]
    ]
    .sort_values('revenue', ascending=False)
    .head(20)
)

In [ ]:
intervention_watchlist_ranked = (
    intervention_watchlist
    .copy()
)

intervention_watchlist_ranked['risk_indicator'] = (
    intervention_watchlist_ranked['late_rate']
    + intervention_watchlist_ranked['low_review_rate']
)

intervention_watchlist_ranked = (
    intervention_watchlist_ranked
    .sort_values(
        ['risk_indicator', 'revenue'],
        ascending=[False, False]
    )
)

print("========== TOP INTERVENTION CANDIDATES ==========")

display(
    intervention_watchlist_ranked[
        [
            'seller_id',
            'revenue',
            'delivered_orders',
            'late_rate',
            'avg_review_score',
            'low_review_rate',
            'seller_state',
            'risk_indicator'
        ]
    ].head(15)
)

In [ ]:
healthy_sellers = seller_master[
    (seller_master['revenue'] > HIGH_REVENUE_THRESHOLD) &
    (seller_master['delivered_orders'] >= MIN_DELIVERED_ORDERS) &
    (seller_master['late_rate'] <= HIGH_LATE_RATE) &
    (seller_master['low_review_rate'] <= HIGH_LOW_REVIEW_RATE)
].copy()

print("========== HEALTHY SELLER BENCHMARKS ==========")
print("Number of healthy sellers:", len(healthy_sellers))

display(
    healthy_sellers[
        [
            'seller_id',
            'revenue',
            'delivered_orders',
            'late_rate',
            'reviewed_orders',
            'avg_review_score',
            'low_review_rate',
            'seller_state'
        ]
    ]
    .sort_values('revenue', ascending=False)
    .head(20)
)

In [ ]:
state_delivery = (
    seller_master[
        seller_master['delivered_orders'] >= MIN_DELIVERED_ORDERS
    ]
    .groupby('seller_state')
    .agg(
        sellers=('seller_id', 'nunique'),
        delivered_orders=('delivered_orders', 'sum'),
        late_orders=('late_orders', 'sum'),
        avg_late_rate=('late_rate', 'mean')
    )
    .reset_index()
)

state_delivery['weighted_late_rate'] = (
    state_delivery['late_orders']
    / state_delivery['delivered_orders']
    * 100
)

print("========== STATE DELIVERY PERFORMANCE ==========")

display(
    state_delivery
    .sort_values('weighted_late_rate', ascending=False)
)

Seller delivery performance varies by geography, but state-level comparisons must be interpreted cautiously because seller counts differ substantially. São Paulo has the largest active seller base (524) and a weighted late rate of 8.57%, while some smaller states show higher rates based on very few sellers. Therefore, geography should be treated as a contextual factor rather than a standalone reason for intervention.

In [ ]:
import matplotlib.pyplot as plt

# Sellers with enough delivered orders
active_sellers = seller_master[
    seller_master['delivered_orders'] >= MIN_DELIVERED_ORDERS
].copy()

plt.figure(figsize=(10, 6))

# All active sellers
plt.scatter(
    active_sellers['revenue'],
    active_sellers['late_rate'],
    alpha=0.35,
    label='Other active sellers'
)

# Intervention candidates
plt.scatter(
    intervention_watchlist['revenue'],
    intervention_watchlist['late_rate'],
    alpha=0.8,
    label='Intervention candidates'
)

# Threshold lines
plt.axvline(
    HIGH_REVENUE_THRESHOLD,
    linestyle='--',
    label='High-revenue threshold'
)

plt.axhline(
    HIGH_LATE_RATE,
    linestyle='--',
    label='High late-rate threshold'
)

plt.xlabel('Seller Revenue')
plt.ylabel('Late Rate (%)')
plt.title('Seller Revenue vs Delivery Late Rate')
plt.legend()
plt.grid(alpha=0.2)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

active_sellers = seller_master[
    seller_master['delivered_orders'] >= MIN_DELIVERED_ORDERS
].copy()

plt.figure(figsize=(10, 6))

plt.scatter(
    active_sellers['revenue'],
    active_sellers['late_rate'],
    alpha=0.30,
    label='Other active sellers'
)

plt.scatter(
    intervention_watchlist['revenue'],
    intervention_watchlist['late_rate'],
    alpha=0.85,
    label='Intervention candidates'
)

plt.axvline(
    HIGH_REVENUE_THRESHOLD,
    linestyle='--',
    label='High-revenue threshold'
)

plt.axhline(
    HIGH_LATE_RATE,
    linestyle='--',
    label='High late-rate threshold'
)

plt.xscale('log')

plt.xlabel('Seller Revenue (log scale)')
plt.ylabel('Late Rate (%)')
plt.title('Seller Revenue vs Delivery Late Rate')

plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

active_sellers = seller_master[
    seller_master['delivered_orders'] >= MIN_DELIVERED_ORDERS
].copy()

# Remove sellers without a review score
active_sellers = active_sellers[
    active_sellers['avg_review_score'].notna()
]

plt.figure(figsize=(10, 6))

# Other active sellers
plt.scatter(
    active_sellers['revenue'],
    active_sellers['avg_review_score'],
    alpha=0.30,
    label='Other active sellers'
)

# Intervention candidates
plt.scatter(
    intervention_watchlist['revenue'],
    intervention_watchlist['avg_review_score'],
    alpha=0.85,
    label='Intervention candidates'
)

# High revenue threshold
plt.axvline(
    HIGH_REVENUE_THRESHOLD,
    linestyle='--',
    label='High-revenue threshold'
)

plt.xlabel('Seller Revenue (log scale)')
plt.ylabel('Average Review Score')
plt.title('Seller Revenue vs Customer Satisfaction')

plt.xscale('log')

plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Keep states with at least 10 active sellers
state_chart = state_delivery[
    state_delivery['sellers'] >= 10
].copy()

# Sort from highest to lowest late rate
state_chart = state_chart.sort_values(
    'weighted_late_rate',
    ascending=True
)

plt.figure(figsize=(10, 6))

plt.barh(
    state_chart['seller_state'],
    state_chart['weighted_late_rate']
)

plt.xlabel('Weighted Late Rate (%)')
plt.ylabel('Seller State')
plt.title('Delivery Performance by Seller State')

plt.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Select top 10 intervention candidates
top_10_intervention = (
    intervention_watchlist_ranked
    .head(10)
    .sort_values('risk_indicator', ascending=True)
)

plt.figure(figsize=(10, 6))

plt.barh(
    top_10_intervention['seller_id'].str[:8],
    top_10_intervention['risk_indicator']
)

plt.xlabel('Operational Risk Indicator')
plt.ylabel('Seller ID')
plt.title('Top 10 Seller Intervention Candidates')

plt.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.show()

# Issue #8 — Seller Performance and Intervention Analysis

## Objective

The objective of this analysis is to identify sellers that contribute significant marketplace value but may require operational intervention because of delivery delays and weaker customer experience.

The analysis also identifies healthy sellers that can be used as benchmark examples and examines whether seller geography is associated with delivery performance.

Methodology

Seller performance was evaluated using four main dimensions:

1. Business contribution — seller revenue and order volume.
2. Delivery performance — delivered orders and late-delivery rate.
3. Customer satisfaction — average review score and low-review rate.
4. Seller geography — seller state and state-level delivery performance.

To avoid unstable comparisons from very small sellers, sellers with at least 20 delivered orders were used for comparative delivery and watchlist analysis.

The thresholds for high revenue, high late rate, and high low-review rate were based on the 75th percentile of the respective distributions.

## Watchlist Criteria and Results

A seller was considered an intervention candidate when all of the following conditions were met:

- Revenue > R$3,280.83 (75th percentile)
- At least 20 delivered orders
- Late rate > 10.69% (75th percentile)
- Low-review rate > 21.43% (75th percentile)

Using these criteria, 50 sellers were identified as intervention candidates.

These sellers combine relatively high marketplace contribution with delivery and customer-experience concerns and should be prioritized for further operational investigation.

## Healthy Seller Benchmark

The analysis identified 435 healthy sellers that met the following conditions:

- Revenue > R$3,280.83
- At least 20 delivered orders
- Late rate ≤ 10.69%
- Low-review rate ≤ 21.43%

These sellers combine strong marketplace contribution with relatively good delivery and customer-experience performance.

They can be used as benchmark examples to understand practices associated with reliable fulfilment and better customer satisfaction.

## Key Findings

1. High-value sellers and delivery risk

The analysis identified sellers with relatively high revenue and above-threshold late-delivery rates. The intervention watchlist contains 50 sellers that also show elevated low-review rates.

2. Customer experience

The intervention candidates generally show weaker average review scores and higher low-review rates than many other active sellers. This suggests that delivery problems and customer-experience concerns can occur together.

3. Healthy sellers

A group of 435 healthy sellers combines relatively strong revenue contribution with better delivery and review performance. These sellers provide useful benchmark examples.

### 4. Geographic variation

Delivery performance varies across seller states. Among states with at least 10 active sellers, SP and RJ have relatively higher weighted late rates, while RS has a lower weighted late rate.

However, geography should be treated as a contextual factor rather than a standalone explanation because seller counts and operating conditions vary across states.

## Business Recommendations

For intervention sellers

- Prioritize high-value sellers that show both delivery delays and weak customer experience.
- Review fulfilment, dispatch and shipping processes for repeated late deliveries.
- Monitor 1–2 star reviews to identify recurring customer complaints.
- Track the performance of intervention sellers periodically after corrective actions.

For healthy sellers

- Study the operational practices of healthy high-value sellers.
- Use these sellers as benchmark examples for fulfilment and customer-service practices.
- Share useful operational practices with sellers requiring improvement.
- Continue supporting healthy sellers so that strong performance is maintained.

Geography

- Use seller geography as a supporting factor when investigating delivery problems.
- Avoid treating a state as inherently high-risk when the number of sellers is very small.

## Limitations

- Seller performance can be unstable when the number of delivered orders is very small. Therefore, a minimum of 20 delivered orders was used for comparative analysis.
- Some seller states have very few active sellers, so state-level comparisons should be interpreted cautiously.
- The intervention watchlist identifies sellers for further investigation; it does not prove that sellers are solely responsible for delivery delays or poor reviews.
- The Operational Risk Indicator used for prioritization is a simple exploratory measure calculated as late rate + low-review rate. It is not a formal statistical risk score.
- Review scores may be affected by factors other than delivery performance, such as product quality, pricing or customer expectations.

## Conclusion

This seller analysis identified important differences in seller contribution, delivery performance and customer experience.

Using data-driven thresholds, 50 sellers were identified as intervention candidates because they combine relatively high revenue with higher late-delivery and low-review rates. At the same time, 435 healthy sellers were identified as potential benchmark examples.

The analysis also shows variation in delivery performance across seller states, although geographic comparisons need to be interpreted carefully because seller volumes differ substantially.

Overall, the findings suggest that the marketplace can improve seller performance by prioritizing high-value problem sellers for operational intervention and using healthy sellers as benchmarks for better fulfilment and customer experience.